In [1]:
import sys
from pathlib import Path

# Add workspace root to Python path
workspace_root = Path().resolve().parent.parent
if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))

# This Script transforms a .ttl file with URIS containing ids to a .txt file in the "G-Care" format


In [2]:
import os
import json
import random
from tqdm import tqdm
from pathlib import Path
import re
from benchmarks.helper import most_common_predicates
from benchmarks.rdf_types import *

In [3]:
# Sets of entities and predicates
vertices = set()
predicates = set()

In [4]:
# Declare which URI corresponds to rdf:type
# rdf_type_uri = "<http://example.com/13000179>" # Yago
# rdf_type_uri = "<http://example.org/1>" # Lubm
# rdf_type_uri = '<http://example.org_derivationally_related_form>' # wn18rr / wn18rr_ind
# rdf_type_uri = '<http://example.orggenre>' # dbpedia100k
# rdf_type_uri = '<http://www.wikidata.org/prop/direct/P31>' # wikidata
# rdf_type_uri = '<http://ex.org/03>' # SWDF
rdf_type_uri = rdf_type_uri_swdf

In [5]:
path = Path("/Users/lukas/Workspace/Uni/BachelorsThesis/FICE/data/swdf/raw/59320634")
out_path = Path('/Users/lukas/Workspace/Uni/BachelorsThesis/FICE/benchmarks/gcare/gcare/data/dataset/swdf')
out_path.mkdir(parents=True, exist_ok=True)
dataset_name = 'swdf'

# rdf_type_uri = most_common_predicates(path)[0][0]

In [6]:
clean_url = rdf_type_uri.strip('<> "\'')
base_uri = clean_url[:clean_url.rfind('/') + 1]

In [7]:
# Add all entities and predicates from the .ttl file
l = 0
ttl_file = open(path, "r")
for line in tqdm(ttl_file):
    atoms = line.split(" ")[:-1]
    #if not atoms == []:
    if True:
        l += 1
        vertices.add(atoms[0])
        vertices.add(atoms[2])
        predicates.add(atoms[1])

242256it [00:00, 2536895.91it/s]


In [8]:
# We need to map the URL ids to entity and predicate ids
id_to_id_mapping = {}
id_to_id_mapping_predicate = {}

In [9]:
# Creating Vertex Dict and save entity mappings
# Create mapping
vertex_dict = {}
vid = 0
for vertex in tqdm(vertices):
    dvid = vertex.split("/")[-1].replace(">", "")
    #vertex_dict[vertex] = [dvid]
    vertex_dict[vertex] = [vid]
    id_to_id_mapping[vertex] = vid
    vid += 1

100%|██████████| 76711/76711 [00:00<00:00, 2130578.12it/s]


In [10]:
# Saving Mappings for Predicates
pid = 0
for p in predicates:
    id_to_id_mapping_predicate[p] = pid
    pid += 1

In [11]:
# Add Vertex types (rdf:type)
ttl_file = open(path, "r")
for line in tqdm(ttl_file):
    atoms = line.split(" ")[:-1]
    # If triple has predicate rdf:type
    if atoms[1] == rdf_type_uri:
        type_id = id_to_id_mapping[atoms[2]]
        vertex_dict[atoms[0]].append(type_id)


242256it [00:00, 3552594.48it/s]


In [12]:
for v in vertex_dict.values():
    if len(v) == 1:
        v.append(0)

In [13]:
# all entreis shoud have length 2 now
all_entries_with_length_2_or_more = all([len(v) >= 2 for v in vertex_dict.values()])
assert all_entries_with_length_2_or_more
print(len(vertex_dict))
print(random.sample(list(vertex_dict.items()), 10))
print(random.sample(list(vertex_dict.keys()), 10))

76711
[('<http://ex.org/16037>', [38245, 60148]), ('<http://ex.org/55743>', [25026, 0]), ('<http://ex.org/6439>', [50727, 29895]), ('<http://ex.org/16857>', [57204, 47850]), ('<http://ex.org/44141>', [32209, 0]), ('<http://ex.org/64055>', [19116, 0]), ('<http://ex.org/25949>', [36168, 63584]), ('<http://ex.org/37486>', [63527, 22338]), ('<http://ex.org/69988>', [12952, 0]), ('<http://ex.org/34743>', [1884, 63584])]
['<http://ex.org/6333>', '<http://ex.org/45202>', '<http://ex.org/33605>', '<http://ex.org/64801>', '<http://ex.org/71852>', '<http://ex.org/33347>', '<http://ex.org/63089>', '<http://ex.org/42756>', '<http://ex.org/27362>', '<http://ex.org/75818>']


In [14]:
# Creating Edge List
edge_list = []
ttl_file = open(path, "r")
for tp in tqdm(ttl_file):
    tp = tp.split(" ")[:-1]
    s = tp[0]
    p = tp[1]
    o = tp[2]
    #edge_label = tp[1].split("/")[-1].replace(">", "") if not "?" in tp[1] else -1
    edge_list.append([vertex_dict[s][0], vertex_dict[o][0], id_to_id_mapping_predicate[p]])

242256it [00:00, 1369212.47it/s]


In [15]:
# Writing the Data File
with open(out_path / f"{dataset_name}.txt", "w") as f:
    f.write("t # 1")
    f.write("\n")
    for v in tqdm(vertex_dict):
        line = "v " + " ".join(map(str, vertex_dict[v]))
        f.write(line)
        f.write("\n")
        # print(line)
    for e in tqdm(edge_list):
        f.write("e " + str(e[0]) + " " + str(e[1]) + " " + str(e[2]))
        f.write("\n")

100%|██████████| 242256/242256 [00:00<00:00, 3329418.72it/s]


In [16]:
# Save to ID to ID mapping for later query transformation
with open(out_path / "id_to_id_mapping.json", "w") as f:
    json.dump(id_to_id_mapping, f)
    
with open(out_path / "id_to_id_mapping_predicate.json", "w") as f:
    json.dump(id_to_id_mapping_predicate, f)

# End 

In [17]:
# Dieser check funktioniert nicht zwischen, da es nicht garantiert ist, dass alle ids
# in edge list, die durch das hochzählen der vertex ids entstanden sind, auch in vertex_dict sind.
# Beispiel: in swdf gibt es nicht http://ex.org/0 als uri in vertex_dict gibt es aber schon einen vid = 0
print(random.sample(edge_list, 10))
for e in edge_list:
    try:
      assert f"<{base_uri}{str(e[0])}>" in vertex_dict
    except:
      print(f"<{base_uri}{str(e[0])}>")
      print(f"Missing edge entity in vertex_dict: {e}")
    try:
      assert f"<{base_uri}{str(e[1])}>" in vertex_dict
    except:
      print(f"<{base_uri}{str(e[1])}>")
      print(f"Missing edge entity in vertex_dict: {e}")
      print(f"vertex_dict: {vertex_dict}")

[[9301, 73324, 16], [12790, 17764, 106], [40227, 72973, 13], [63101, 47850, 1], [3520, 75744, 20], [62874, 49063, 16], [9417, 1307, 96], [16299, 59701, 60], [23969, 41549, 38], [15231, 28969, 13]]
<http://ex.org/0>
Missing edge entity in vertex_dict: [2386, 0, 43]
vertex_dict: {'<http://ex.org/10346>': [0, 0], '<http://ex.org/19640>': [1, 0], '<http://ex.org/10882>': [2, 0], '<http://ex.org/24890>': [3, 0], '<http://ex.org/20474>': [4, 0], '<http://ex.org/22783>': [5, 63584], '<http://ex.org/6738>': [6, 0], '<http://ex.org/36060>': [7, 0], '<http://ex.org/10921>': [8, 0], '<http://ex.org/20337>': [9, 0], '<http://ex.org/30175>': [10, 60148], '<http://ex.org/5326>': [11, 0], '<http://ex.org/10932>': [12, 0], '<http://ex.org/10634>': [13, 0], '<http://ex.org/8010>': [14, 47658], '<http://ex.org/15710>': [15, 0], '<http://ex.org/37889>': [16, 22338], '<http://ex.org/15116>': [17, 47850], '<http://ex.org/38830>': [18, 0], '<http://ex.org/9124>': [19, 0], '<http://ex.org/61275>': [20, 0], '